# PAUT SCN-Attention U-Net — train on a free Colab GPU

Your laptop is CPU-only, so train here on a free GPU, then download the checkpoint and run
inference / the dashboard locally. Everything is one small zip — no Git, no Drive needed.

### Do this first
**Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4)**, then Save.

Then run the cells top to bottom. Steps: check GPU -> upload zip -> install -> preprocess ->
pseudo-labels -> train -> download checkpoint.

In [ ]:
#@title 1. Confirm we have a GPU
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('NO GPU — go to Runtime > Change runtime type > GPU, then re-run this cell.')

In [ ]:
#@title 2. Upload paut-digital-twin.zip (from your Downloads folder)
# A 'Choose Files' button appears below — pick paut-digital-twin.zip.
from google.colab import files
import zipfile, os
up = files.upload()
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall('.')
os.chdir('paut-digital-twin')
print('now in:', os.getcwd())
print('contents:', sorted(os.listdir('.')))

In [ ]:
#@title 3. Install dependencies (torch is already on Colab)
!pip -q install kymatio scikit-image opencv-python-headless pyyaml
print('done')

In [ ]:
#@title 4. Preprocess: raw images -> normalized patches + manifest + split
!python -m src.data.build_dataset --config configs/preprocess.yaml

In [ ]:
#@title 5. Stage-1 pseudo-label masks
!python -m src.data.build_pseudo_labels --config configs/pseudo_label.yaml

In [ ]:
#@title 6. (recommended on GPU) speed up training: bigger batch, mixed precision
import yaml
with open('configs/model.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['train']['batch_size'] = 16      # GPU has plenty of memory
cfg['train']['amp'] = True           # mixed precision = faster on GPU
cfg['data']['num_workers'] = 2       # parallel data loading
with open('configs/model.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('updated: batch_size=16, amp=True, num_workers=2')

In [ ]:
#@title 7. Train (full run). Watch val dice_fg / cls_acc; best checkpoint saved automatically.
# Tip: do a quick sanity check first with  --limit 40 --epochs 3  if you want.
!python -m src.models.train --config configs/model.yaml

In [ ]:
#@title 8. (optional) plain U-Net baseline for the data-ablation comparison
# !python -m src.models.train --config configs/model.yaml --no-scattering

In [ ]:
#@title 9. Download the trained checkpoint to your laptop
from google.colab import files
import glob
ckpts = glob.glob('checkpoints/*_best.pt')
print('found:', ckpts)
for f in ckpts:
    files.download(f)

### After training
Put the downloaded `*_best.pt` into a `checkpoints/` folder in your local project. The next
milestones (characterization, XAI, dashboard) load it for inference on CPU.

**If Colab disconnects** during a long run: re-run cells 1-5 (and 6), then 7. The split is fixed
by seed, so it is reproducible. Keep the tab active; free Colab disconnects when idle.